# PhishDrift milestone 1 on Colab

Runs the pooled-corpus build on Colab's disk and network, so your own PC can stay off.
Run the cells **in order** and keep this browser tab open until the last cell finishes
(free Colab sessions end after about 12 hours, or earlier if the tab is closed or idle).

Needs a Google account with Drive (a few hundred MB for the results). Nothing is committed to GitHub;
datasets stay on Colab's disk and are deleted when the session ends.

## 1. Disk check and Drive

In [ ]:
!df -h /content | tail -1
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/phishdrift_results'
os.makedirs(OUT, exist_ok=True)
print('results will be copied to', OUT)

## 2. Get the code and install (about 2 minutes)

In [ ]:
%%bash
set -e
cd /content
rm -rf phishdrift
git clone --depth 1 -b claude/project-thread-q5f1ns https://github.com/gangwaritvik/phishdrift.git
cd phishdrift
pip install -q -e ".[dev,datasets]"
python --version

## 3. Tests (expect all green)

In [ ]:
%%bash
cd /content/phishdrift
python -m pytest -q 2>&1 | tail -5

## 4. Optional: manual sources
Phish360, Phish-Blitz, Phishpedia 30k, URL-Phish and PhishStorm are manual downloads (see the README table).
If you have them, upload them to Drive and copy them into `/content/phishdrift/data/raw/<source>/` here.
Skipping this is fine: the build skips missing sources and says which.

In [ ]:
# Example (edit the Drive path); leave commented if you have none:
# !mkdir -p /content/phishdrift/data/raw/urlphish && cp /content/drive/MyDrive/urlphish/*.csv /content/phishdrift/data/raw/urlphish/

## 5. Collect live feeds (about 30 minutes, mostly the benign crawl)

In [ ]:
%%bash
cd /content/phishdrift
mkdir -p logs
python -u -m collector run --no-capture 2>&1 | tee logs/collect.log

## 6. First build: downloads PhreshPhish one file at a time (hours)
Each file is deleted after it is read, and the loader checks that every row was read.
Leave this cell running. The log shows `[n/77] rows read` as it goes.

In [ ]:
%%bash
cd /content/phishdrift
python -u -m data_pipeline build 2>&1 | tee logs/build1.log

## 7. Seeds, PhiUSIIL crawl, second build

In [ ]:
%%bash
cd /content/phishdrift
python -u -m data_pipeline seeds 2>&1 | tee logs/seeds.log
python -u -m collector crawl --seeds phiusiil 2>&1 | tee logs/crawl.log
python -u -m data_pipeline build --reuse-interim --reload phiusiil urlphish live 2>&1 | tee logs/build2.log

## 8. Save results to Drive and show the data card

In [ ]:
%%bash
cd /content/phishdrift
cp -r reports logs /content/drive/MyDrive/phishdrift_results/
du -sh data
ls -la reports

In [ ]:
print(open('/content/phishdrift/reports/data_card.md').read())